# 순열 중요도 (Permutation Importance) 분석
- **모델**: 최종 모델 ID+FIN+MKT+EMO (vol3d, ±0.7%)
- **설정**: tune_v2 최적값 (lr=0.01, depth=3, leaf=20, iter=300, l2=0.0)
- **방법**: Holdout 세트 기준 Macro-F1 감소량, n_repeats=30
- **출력**: feature_importance.csv + top_importance.png (논문 Figure용)
- HistGradientBoosting은 내장 importance가 없어 permutation importance 사용

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, make_scorer

DATA_DIR     = './'
HOLDOUT_DAYS = 8
LABEL_COL    = 'label_07'

FIN_COLS = [
    'return_1d','return_3d','return_5d','volatility_3d',
    'volume_change','volume_ma_ratio','sector_return_mean',
    'relative_return_to_sector','relative_return_to_market',
    'relative_volatility_to_sector_3d'
]
MKT_COLS = [
    'VIX_Close','VIX_return_1d','VIX_change_3d',
    'SPY_return_1d','SPY_return_3d','QQQ_return_3d',
    'Oil_return_1d','Gold_return_1d','Dollar_return_1d','Treasury10Y_return_1d'
]

In [ ]:
# ── 라벨 계산 ──
raw = pd.read_csv(DATA_DIR + 'israel_hamas_FIN_MKT_features(step3).csv')
raw.columns = raw.columns.str.strip()
raw['Date'] = pd.to_datetime(raw['Date'])
raw = raw.sort_values(['ticker','Date'])
raw['next_return'] = raw.groupby('ticker')['Close'].pct_change(1).shift(-1)
raw['label_07'] = np.nan
raw.loc[raw['next_return'] >  0.007, 'label_07'] = 2
raw.loc[raw['next_return'] < -0.007, 'label_07'] = 0
raw.loc[(raw['next_return'] >= -0.007) & (raw['next_return'] <= 0.007), 'label_07'] = 1
labels = raw[['Date','ticker','label_07']]
print('라벨 계산 완료')

In [ ]:
# ── 데이터 로드 & 피처 구성 ──
df = pd.read_csv(DATA_DIR + 'model_input_dataset_3day_volatility(step3).csv', parse_dates=['Date'])
df = pd.merge(df, labels, on=['Date','ticker'], how='left')

emo_cols = [c for c in df.columns if c.startswith('emo_')]
# sector_return_mean 은 FIN 피처이므로 ID(OHE)에서 제외
id_cols  = [c for c in df.columns
            if (c.startswith('sector_') or c.startswith('ticker_'))
            and c != 'sector_return_mean']

# 중복 제거(순서 유지)
raw_feats = id_cols + FIN_COLS + MKT_COLS + emo_cols
seen = set()
FEAT_COLS = [c for c in raw_feats if not (c in seen or seen.add(c))]

assert len(FEAT_COLS) == len(set(FEAT_COLS)), '중복 피처 존재!'
print(f'피처 {len(FEAT_COLS)}개  (ID:{len(id_cols)} FIN:{len(FIN_COLS)} MKT:{len(MKT_COLS)} EMO:{len(emo_cols)})')

sub = df[FEAT_COLS + [LABEL_COL,'Date']].dropna(subset=[LABEL_COL])
dates = sorted(sub['Date'].unique())
train = sub[sub['Date'].isin(dates[:-HOLDOUT_DAYS])]
hold  = sub[sub['Date'].isin(dates[-HOLDOUT_DAYS:])]

X_tr = train[FEAT_COLS].values; y_tr = train[LABEL_COL].astype(int).values
X_ho = hold[FEAT_COLS].values;  y_ho = hold[LABEL_COL].astype(int).values
print(f'Train {len(X_tr)}행 / Holdout {len(X_ho)}행')

In [ ]:
# ── 최종 모델 훈련 (tune_v2 최적 설정) ──
model = HistGradientBoostingClassifier(
    learning_rate=0.01, max_depth=3,
    min_samples_leaf=20, max_iter=300,
    l2_regularization=0.0, class_weight='balanced',
    random_state=42
)
model.fit(X_tr, y_tr)
ho_f1 = f1_score(y_ho, model.predict(X_ho), average='macro', zero_division=0)
print(f'Holdout Macro-F1: {ho_f1:.4f}')

In [ ]:
# ── Permutation Importance (Holdout 기준, n_repeats=30) ──
macro_f1 = make_scorer(f1_score, average='macro', zero_division=0)
result = permutation_importance(
    model, X_ho, y_ho,
    scoring=macro_f1, n_repeats=30, random_state=42, n_jobs=-1
)

def group_of(f):
    if f.startswith('emo_'): return 'EMO'
    if f.startswith('sector_') or f.startswith('ticker_'): return 'ID'
    if f in FIN_COLS: return 'FIN'
    if f in MKT_COLS: return 'MKT'
    return 'OTHER'

imp = pd.DataFrame({
    'feature': FEAT_COLS,
    'importance': result.importances_mean,
    'std': result.importances_std,
})
imp['group'] = imp['feature'].map(group_of)
imp = imp.sort_values('importance', ascending=False).reset_index(drop=True)
imp.to_csv(DATA_DIR + 'feature_importance.csv', index=False)

print('── Top 15 피처 ──')
print(imp.head(15)[['feature','group','importance','std']].to_string(index=False))

In [ ]:
# ── 그룹별 중요도 합계 ──
grp = imp.groupby('group')['importance'].sum().sort_values(ascending=False)
print('── 그룹별 importance 합 ──')
print(grp.to_string())
print()
# Top10 중 그룹 분포
top10_grp = imp.head(10)['group'].value_counts()
print('── Top 10 피처의 그룹 분포 ──')
print(top10_grp.to_string())

In [ ]:
# ── 논문 Figure: Top 15 막대그래프 (그룹별 색상) ──
top = imp.head(15).iloc[::-1]  # 위에서 아래로 큰 값
color_map = {'FIN':'#2C3E50','MKT':'#2980B9','EMO':'#E74C3C','ID':'#95A5A6','OTHER':'#BDC3C7'}
colors = [color_map[g] for g in top['group']]

fig, ax = plt.subplots(figsize=(9,6))
ax.barh(top['feature'], top['importance'], xerr=top['std'],
        color=colors, edgecolor='black', linewidth=0.4, error_kw={'lw':0.7})
ax.set_xlabel('Permutation Importance (Holdout Macro-F1 decrease)', fontsize=11)
ax.set_title('Top 15 Feature Importance (n_repeats=30)', fontsize=12)

# 범례
from matplotlib.patches import Patch
legend = [Patch(facecolor=color_map[g], label=g) for g in ['FIN','MKT','EMO','ID']]
ax.legend(handles=legend, loc='lower right', title='Feature Group')
plt.tight_layout()
plt.savefig(DATA_DIR + 'top_importance.png', dpi=200, bbox_inches='tight')
plt.show()
print('저장: top_importance.png, feature_importance.csv')